In [ ]:
!pip install -q langgraph langchain-core langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [1]:
import langgraph, langchain_core
from importlib.metadata import version
print("langgraph:", version("langgraph"))
print("langchain-core:", langchain_core.__version__)
print("langchain-google-genai:", version("langchain-google-genai"))

langgraph: 1.2.11
langchain-core: 1.6.3
langchain-google-genai: 4.4.0


In [17]:
import langgraph, langchain_core
from dotenv import load_dotenv
import os

load_dotenv()
print("langgraph OK, dotenv OK")
print("key loaded:", os.getenv("RESEARCHER_API_KEY") is not None)

langgraph OK, dotenv OK
key loaded: True


In [18]:
# Loading API Key For Colab
# from google.colab import userdata
# import os
# key_names = ["Researcher", "Writer", "Critic"]
# for name in key_names:
#   val = userdata.get(name)
#   os.environ[name] = val
#   print(f"{name}: {'OK, len=' + str(len(val)) if val else 'MISSING'}")

In [32]:
from typing import TypedDict, List
class ResearchState(TypedDict):
  topic: str            # User input topic
  research_notes: str   # Researcher generate info clearn up
  draft: str            # Writer current draft
  critique: str         # Critic latest judge opinion
  revision_count: int   # Current correctness times, for controling loop ending
  approved: bool        # Critic passing or not

print("State Def result complete")

State Def result complete


In [20]:
# Loading Gemini Model
from langchain_google_genai import ChatGoogleGenerativeAI
researcher_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["RESEARCHER_API_KEY"],
)

# Writer & Critic ideally use 3.5-flash
#  Due to the daily 20 limitation for 3.5-flash
#  , use 3.5-flash-lite instead

# writer_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     google_api_key=os.environ["Writer"],
# )

# critic_llm = ChatGoogleGenerativeAI(
#     model="gemini-3.5-flash",
#     google_api_key=os.environ["Critic"],
# )

writer_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["WRITER_API_KEY"],
)

critic_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.environ["CRITIC_API_KEY"],
)



In [21]:
# Assistant Function

def extract_text(response) -> str:
  """
    Dealing with Gemini return format (new version could be list of dict, or number)
  """
  content = response.content
  if isinstance(content, str):
    return content
  if isinstance(content, list):
    return "".join(
        block.get("text", "")
        for block in content
        if isinstance(block,dict)
    )
  return str(content)

In [22]:
# Research Node Define
def researcher_node(state: ResearchState) -> dict:
  prompt = (
      f"You are a research assistant, that focuing on topic '{state['topic']}', "
      f"List out 5 to 8 key facts, background, or anything need to concider with clear, high volume info"
      "Don't include unnecessary details."
  )
  response = researcher_llm.invoke(prompt)
  return {"research_notes": extract_text(response)}

# Quick test
test_state = {"topic": "Current Taiwan electrical motorbike situation",
              "research_notes": "",
              "draft": "",
              "critique": "",
              "revision_count": 0,
              "approved": False}
result = researcher_node(test_state)
print(type(result["research_notes"]))
print(result["research_notes"][:200])


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


<class 'str'>
Here are 7 high-volume, key facts and background considerations regarding the current Taiwan electric motorbike (e-scooter) situation:

1. **Market Dominance of Gogoro and the Battery-Swapping Standar


In [23]:
# Writer Node Define
def writer_node(state: ResearchState) -> dict:
  if state.get("critique"):
    # Modification Loop: regarding to Critic to modify previous draft
    prompt = (
        f"This is the draft that you wrote regarding to the topic '{state['topic']}'"
        f"Critiquer gave the opinion as follow: \n{state['critique']}\n\n"
        f"Please modify the draft, and output the corrected result."
        "Not only what you changed, but the complete draft"
    )
  else:
    # First Loop: draft according to the topic
    prompt = (
        f"You are a professional writer. Regarding to the topic: '{state['topic']}'"
        f"Write a clear structured, 300-500 words first draft: \n\n{state['research_notes']}"
    )
  response = writer_llm.invoke(prompt)
  return {
      "draft": extract_text(response),
      "revision_count": state["revision_count"] + (1 if state.get("critique") else 0),
  }

# Quick test follow the previous research_notes
test_state["research_notes"] = result["research_notes"]
draft_result = writer_node(test_state)
print(draft_result["draft"][:300])
print("revision_count:", draft_result["revision_count"])

**Title:** Navigating Taiwan’s Electric Motorbike Revolution: Current Landscape and Future Outlook

Taiwan is globally recognized as a powerhouse for two-wheeled transportation. As the island nation pivots toward a greener future, the electric motorbike (e-scooter) sector has emerged as a critical b
revision_count: 0


In [35]:
# Critic Node Define
def critic_node(state: ResearchState) -> dict:
    prompt = (
        f"You are a strict checker, check the report draft regarding to the topic:'{state['topic']}':\n\n"
        f"{state['draft']}\n\n"
        f"Check if content is accurate, clear, or if there's any obvious missing point or logic issue.\n"
        f"Please use the following format to reply (first line must be APPROVED or REVISE, not other word in the first line)\n"
        f"APPROVED or REVISE\n"
        f"The following should start the opinion (if it is APPROVED, brifly explain why)"
    )
    response = critic_llm.invoke(prompt)
    text = extract_text(response).strip()

    lines = text.split("\n", 1)
    verdict = lines[0].strip().upper()
    feedback = lines[1].strip() if len(lines) > 1 else ""

    return {
        "approved": verdict.startswith("APPROVED"),
        "critique": feedback,
        "revision_count": state["revision_count"]
    }

# Quick test: use draft from previous quick test
test_state["draft"] = draft_result["draft"]
critique_result = critic_node(test_state)
print("approved:", critique_result["approved"])
print("critique:", critique_result["critique"][:300])

approved: False
critique: The report is generally well-written, engaging, and covers the core aspects of Taiwan's e-scooter landscape well. However, there are a few factual inaccuracies, outdated market nuances, and a missing macroeconomic perspective that must be addressed before final approval:

1. **Market Share Inaccurac


In [36]:
from langgraph.graph import StateGraph, START, END

MAX_REVISIONS = 3  # Temporary early stop

# Condition edge for critic
def route_after_critic(state: ResearchState) -> str:
    if state["approved"] or state["revision_count"] >= MAX_REVISIONS:
        return "end"
    return "revise"

graph_builder = StateGraph(ResearchState)
# add three nodes for researcher, writer, and critic
graph_builder.add_node("researcher", researcher_node)
graph_builder.add_node("writer", writer_node)
graph_builder.add_node("critic", critic_node)

# add edges connect three state nodes
graph_builder.add_edge(START, "researcher")
graph_builder.add_edge("researcher", "writer")
graph_builder.add_edge("writer", "critic")

# If approved or reach MAX_REVISIONS -> END state
# Eles -> revise
graph_builder.add_conditional_edges(
    "critic",
    route_after_critic,
    {"end": END, "revise": "writer"},
)

graph = graph_builder.compile()
print("Graph Complete Edit")

Graph Complete Edit


In [37]:
initial_state = {
    "topic": "AI replacing all researchers",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

final_state = graph.invoke(initial_state)

print("=== Result ===")
print("Approved:", final_state["approved"])
print("Fix round(s):", final_state["revision_count"])
print("\n=== Final Draft ===")
print(final_state["draft"])
print("\n=== Final Critic Opinion ===")
print(final_state["critique"])

=== Result ===
Approved: True
Fix round(s): 1

=== Final Draft ===
Here is the complete, revised draft incorporating all the critiquer's points:

***

# The Ghost in the Laboratory: Will AI Replace All Researchers?

The image of the scientist—hunched over a microscope, scribbling complex equations on a chalkboard, or pacing nervously waiting for an experiment to run—is deeply embedded in our cultural imagination. For centuries, scientific discovery has been viewed as the ultimate bastion of human ingenuity. Yet, as Large Language Models (LLMs), automated synthesis platforms, and AI-driven hypothesis generators advance at a breathless pace, a provocative question looms over academia and industry alike: Will artificial intelligence eventually replace human researchers entirely?

The short answer is no. While AI will fundamentally transform *how* research is conducted, automating routine tasks and accelerating data processing, the complete displacement of human researchers is unlikely—not

In [38]:
initial_state = {
    "topic": "AI Agent in Enterprise, how employee prevent to be laid off",
    "research_notes": "",
    "draft": "",
    "critique": "",
    "revision_count": 0,
    "approved": False,
}

for event in graph.stream(initial_state):
    for node_name, node_output in event.items():
        print(f"--- State: {node_name} ---")
        if "approved" in node_output:
            print(f"  approved={node_output['approved']}, revision_count={node_output['revision_count']}")
        elif "draft" in node_output:
            print(f"  draft (First 80 char): {node_output['draft'][:80]}...")
        elif "research_notes" in node_output:
            print(f"  research_notes (First 80 char): {node_output['research_notes'][:80]}...")
        print()

--- State: researcher ---
  research_notes (First 80 char): Here are 6 key facts, background contexts, and critical considerations regarding...

--- State: writer ---
  draft (First 80 char): **Navigating the AI Agent Era: How Employees Can Future-Proof Their Careers**

T...

--- State: critic ---
  approved=False, revision_count=0

--- State: writer ---
  draft (First 80 char): Here is the complete, revised draft incorporating the critique. Changes include ...

--- State: critic ---
  approved=True, revision_count=1



In [39]:
def run_research_pipeline(topic: str) -> dict:
    initial_state = {
        "topic": topic,
        "research_notes": "",
        "draft": "",
        "critique": "",
        "revision_count": 0,
        "approved": False,
    }
    final_state = graph.invoke(initial_state)
    return {
        "topic": topic,
        "final_draft": final_state["draft"],
        "approved": final_state["approved"],
        "revision_count": final_state["revision_count"],
        "final_critique": final_state["critique"],
    }

# Testing
output = run_research_pipeline("How junior SWE can get hired after laid off in 2026")
print("approved:", output["approved"])
print("revision_count:", output["revision_count"])
print(output["final_draft"][:200])

approved: True
revision_count: 1
# Navigating the Shift: How Junior Software Engineers Can Get Hired After Being Laid Off in 2026

Being laid off as a junior software engineer in 2026 is a daunting experience. The tech industry has u


In [40]:
# Use gradio to build a simple UI
import gradio as gr

def gradio_handler(topic):
    if not topic.strip():
        return "Please enter the research topic", "", ""
    output = run_research_pipeline(topic)
    status = "✅ Approved" if output["approved"] else "⚠️ Meet the max revise rounds, Not Approved"
    meta = f"{status}| revision count:{output['revision_count']}"
    return meta, output["final_draft"], output["final_critique"]

demo = gr.Interface(
    fn=gradio_handler,
    inputs=gr.Textbox(label="Research Topic", placeholder="ex: Battery factory in 2026"),
    outputs=[
        gr.Textbox(label="Status"),
        gr.Markdown(label="Final Report"),
        gr.Textbox(label="Critic Final Opinion"),
    ],
    title="Multi-Agent Research Assistants",
    description="Researcher → Writer → Critic Cooperate Research report (Gemini 3.5 Flash-Lite)",
)

demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


In [41]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Using stub tool to verify the LLM judgement activity
@tool
def web_search(query: str) -> str:
    """
    Search the updated info when:
    1. Most current truth data needed
    2. Data is outside of your training data
    """
    return f"[STUB] Searching (pretend): {query}"

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=os.getenv("RESEARCHER_API_KEY"),
)
llm_with_tools = llm.bind_tools([web_search])

# Test with two scenario:
# 1. Need to search online
# 2. No need to search
test_prompts = [
    "What's the weather in San Jose today?",           # Need realtime info: calling tool
    "What is 1 + 1 ?",                 # No need to search, not calling tool
]

for p in test_prompts:
    response = llm_with_tools.invoke(p)
    print(f"Prompt: {p}")
    print(f"Tool calls: {response.tool_calls}")
    print(f"Content: {response.content}")
    print("---")

Prompt: What's the weather in San Jose today?
Tool calls: [{'name': 'web_search', 'args': {'query': 'weather in San Jose today'}, 'id': 'call_5277021', 'type': 'tool_call'}]
Content: []
---
Prompt: What is 1 + 1 ?
Tool calls: []
Content: [{'type': 'text', 'text': '1 + 1 = 2', 'extras': {'signature': 'El4KXAERTTIP71x4Ir3IfUAZJVvebzmUVc3BOVbr75Et7+IhxQ6Ma06yjAXm7hiGRKYMKI6y6FnYJ5FWzNz2CTi6SKVD4iG+m42VutSV5j+38iZYv0NqANTT6zCh7893'}}]
---


In [42]:
from langchain_tavily import TavilySearch
import os

web_search = TavilySearch(
    max_results=3,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

# Verify: calling tool (Not connect to LLM), to confirm the return format
result = web_search.invoke({"query": "San Jose weather today"})
print(result)

ValidationError: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={'tavily_api_key': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error